<div style="font-family:-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif;background:#0d1b2a;border-radius:10px;padding:18px 22px;color:#e6edf3;">
  <div style="display:flex;align-items:center;gap:12px;">
    <div style="width:6px;height:36px;background:#3fb950;border-radius:3px;flex:none;"></div>
    <div>
      <div style="font-size:11px;letter-spacing:2px;color:#8b98a5;">NOTEBOOK 03</div>
      <div style="font-size:21px;font-weight:700;line-height:1.15;">Modeling</div>
    </div>
  </div>
  <div style="font-size:13px;color:#a9b4c0;margin-top:9px;">Train a majority-class floor, an interpretable logistic baseline, and tree ensembles. Compare on a time-based hold-out.</div>
</div>

**Protocol.**

- **Split by time:** the earliest 75% of events train, the most recent 25% test. A random split would leak future market regimes into training.
- **Models:** a majority-class floor (sanity check), a scaled **logistic regression** baseline, and **random forest** / **gradient boosting** as nonlinear comparisons.
- **Class weights** balance the moderate class imbalance.
- Everything is seeded for reproducibility.

In [1]:
import sys, os
# Robust: find the project root (folder containing src/) from anywhere
_root = os.path.abspath(os.getcwd())
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, "src")):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.config import CONFIG
from src.plotting import set_style, save_fig
set_style()
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

# Data source: live Yahoo Finance via yfinance.
SOURCE = "yfinance"

In [2]:
from src.config import DATA_PROCESSED
from src.labeling import time_split
from src.modeling import train_all, feature_importance
import joblib

events = pd.read_csv(DATA_PROCESSED / "events.csv", parse_dates=["date"])
train, test = time_split(events, CONFIG.model.test_fraction)
print(f"train={len(train)} ({train.date.min().date()}->{train.date.max().date()})")
print(f"test ={len(test)} ({test.date.min().date()}->{test.date.max().date()})")
print(f"train reversion rate={train.reverted.mean():.3f}  test={test.reverted.mean():.3f}")

train=163 (2020-01-17->2024-11-06)
test =55 (2024-11-22->2026-04-27)
train reversion rate=0.196  test=0.236


## Train and compare

In [3]:
from src.evaluate import metrics_table
fitted, preds, base_rate = train_all(train, test)
print(f"base rate (train P[revert]) = {base_rate:.3f}")
results = metrics_table(test['reverted'].values, preds)
results

base rate (train P[revert]) = 0.196


,accuracy,precision,recall,f1,roc_auc,pr_auc
model,,,,,,
base_rate,0.764,0.000,0.000,0.000,0.500,0.236
logistic,0.745,0.429,0.231,0.300,0.621,0.337
random_forest,0.691,0.250,0.154,0.190,0.553,0.341
gradient_boosting,0.709,0.000,0.000,0.000,0.560,0.281


In [4]:
from src.evaluate import metrics_table as evaluate_metrics
from src.evaluate import backtest

## Robustness: time-series cross-validation on the training set

A single split can be lucky. A blocked, forward-chaining `TimeSeriesSplit` on the training data gives a more stable read on ranking ability (ROC-AUC).

In [5]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from src.modeling import build_models
feat = list(CONFIG.model.feature_cols)
Xtr, ytr = train[feat].values, train['reverted'].values
tscv = TimeSeriesSplit(n_splits=4)
cv = {name: cross_val_score(m, Xtr, ytr, cv=tscv, scoring='roc_auc').mean()
      for name, m in build_models().items()}
pd.Series(cv, name='cv_roc_auc').round(3).sort_values(ascending=False)

logistic            0.565
gradient_boosting   0.547
random_forest       0.506
Name: cv_roc_auc, dtype: float64

## Does the regime feature engineering help?

An ablation: train the same models on the original spread-only features (11)
versus the full set that adds the momentum, detachment, local-cointegration and
sector features (20). If the regime block earns its place, the richer set should
rank events at least as well, especially at the selective end used for trading.

In [8]:
old_features = ['abs_z','z_velocity','spread_vol','half_life','spread_slope',
                'corr_recent','beta','coint_pvalue','market_vol','ret_a_5d','ret_b_5d']
new_features = list(CONFIG.model.feature_cols)

rows = []
for label, feats in [('spread-only (11)', old_features), ('+ regime (20)', new_features)]:
    f_, p_, _ = train_all(train, test, features=feats)
    m_ = evaluate_metrics(test['reverted'].values, p_)
    b_ = m_.drop(index='base_rate')['roc_auc'].idxmax()
    bt = backtest(test, p_[b_]['proba'], 0.60)
    rows.append({'feature_set': label, 'best_model': b_,
                 'roc_auc': m_.loc[b_,'roc_auc'], 'pr_auc': m_.loc[b_,'pr_auc'],
                 'f1': m_.loc[b_,'f1'],
                 'winrate_filtered@0.6': bt['winrate_model'], 'uplift': bt['winrate_uplift']})
pd.DataFrame(rows).set_index('feature_set').round(3)

,best_model,roc_auc,pr_auc,f1,winrate_filtered@0.6,uplift
feature_set,,,,,,
spread-only (11),logistic,0.701,0.375,0.400,0.375,0.139
+ regime (20),logistic,0.621,0.337,0.300,0.400,0.164


## Save the best model

In [9]:
best = results.drop(index='base_rate')['roc_auc'].idxmax()
from src.config import MODELS
joblib.dump(fitted[best], MODELS / f'{best}.joblib')
print('best model:', best, '-> saved to models/')
feature_importance(fitted[best], feat).round(3)

best model: logistic -> saved to models/


spread_vol            1.217
detach_max            0.947
vix_change            0.899
comembership          0.826
abs_z                 0.533
half_life             0.506
corr_recent           0.470
sector_dispersion     0.428
coint_pvalue          0.395
beta                  0.387
corr_baseline         0.373
z_rank                0.355
hurst                 0.326
ret_b_5d              0.308
spread_slope          0.296
n_pairs_extreme       0.294
adf_stat              0.235
corr_change           0.225
coint_recent_pvalue   0.199
market_vol            0.165
vix_level             0.165
ret_a_5d              0.136
vol_spike_max         0.121
mom_rel               0.071
mom_a_252             0.038
mom_b_252             0.022
vol_ratio             0.013
z_velocity            0.001
dtype: float64

## Takeaways

- The logistic baseline sits close to chance on ROC, confirming the problem is genuinely hard and linear effects are weak.
- The **tree ensembles rank events better than the floor**, and the ranking holds up under time-series cross-validation.
- The best model is carried into `04_model_evaluation` for a full read-out and a business backtest.